# LeetCode 2810 — Faulty Keyboard bằng Python

Link đề: https://leetcode.com/problems/faulty-keyboard/ · Độ khó: **Easy** · Chủ đề: String, Simulation, Deque

Skeleton LeetCode:

```python
class Solution:
    def finalString(self, s: str) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **mô phỏng (simulation)**: gõ từng ký tự của `s` lên một bàn phím hỏng; mỗi lần gõ chữ `'i'` thì **toàn bộ chuỗi đã gõ bị đảo ngược**, các chữ khác gõ bình thường.
2. Biết **tự tính tay** từng bước gõ, nhìn chuỗi thay đổi sau mỗi ký tự trước khi viết code.
3. Học các công cụ Python: nối chuỗi `+`, cắt lát đảo ngược `[::-1]`, `list.append` / `list.reverse`, `collections.deque` (`append`, `appendleft`), `str.split` và `"".join`.
4. Đi từ **mô phỏng bằng chuỗi (đảo thật mỗi lần, `O(n²)`)** → **dùng list** → **deque + cờ hướng (không đảo thật, `O(n)`)** → **tách theo `'i'` bằng `split`**.
5. Hiểu mẹo then chốt: **thay vì đảo thật, chỉ nhớ "đang gõ vào đầu hay vào cuối"**, và tránh các lỗi kinh điển: quên chữ `'i'` không được ghi vào chuỗi, đảo nhầm thứ tự, hiểu sai tiền tố đã gõ.

## 2. Đề bài & ví dụ

Bàn phím laptop của bạn bị hỏng: mỗi khi bạn gõ ký tự `'i'`, nó **đảo ngược** chuỗi bạn đã gõ cho tới lúc đó. Gõ các ký tự khác thì hoạt động bình thường (ký tự được **nối vào cuối** chuỗi).

Bạn được cho một chuỗi `s` (0-indexed), bạn gõ lần lượt từng ký tự của `s` lên bàn phím hỏng này.

Trả về **chuỗi cuối cùng** trên màn hình laptop.

Lưu ý: chính ký tự `'i'` **không** xuất hiện trên màn hình; nó chỉ gây ra việc đảo ngược.

| Ví dụ | `s` | Output | Giải thích |
|---|---|---|---|
| 1 | `"string"` | `"rtsng"` | Gõ `s`,`t`,`r` → `"str"`. Gõ `i` → đảo thành `"rts"`. Gõ `n`,`g` → `"rtsng"`. |
| 2 | `"poiinter"` | `"ponter"` | `"p"`, `"po"`, gõ `i` → `"op"`, gõ `i` lần nữa → `"po"`, rồi `n`,`t`,`e`,`r` → `"ponter"`. |

Constraints:

- `1 <= s.length <= 100`
- `s` chỉ gồm các chữ cái thường tiếng Anh.
- `s[0] != 'i'`

In [1]:
# Chạy thử "bằng tay" ví dụ 1: s = "string"
screen = ""
for ch in "string":
    if ch == "i":
        screen = screen[::-1]
    else:
        screen = screen + ch
    print(f"gõ {ch!r:4} -> màn hình: {screen!r}")

gõ 's'  -> màn hình: 's'
gõ 't'  -> màn hình: 'st'
gõ 'r'  -> màn hình: 'str'
gõ 'i'  -> màn hình: 'rts'
gõ 'n'  -> màn hình: 'rtsn'
gõ 'g'  -> màn hình: 'rtsng'


## 3. Kiến thức nền

### 3.1. Chuỗi (`str`), chỉ số và nối chuỗi

Chuỗi là dãy ký tự. `s[0]` là ký tự đầu, `s[-1]` là ký tự cuối. Phép `+` nối hai chuỗi thành **chuỗi mới** (chuỗi trong Python **bất biến**: không sửa tại chỗ được, mỗi lần nối tạo ra bản sao mới).

In [2]:
s = "abc"
print(s[0], s[-1])          # a c
print(s + "d")              # abcd
t = s
s = s + "d"
print(s, t)                 # abcd abc : t vẫn giữ giá trị cũ vì chuỗi bất biến

a c
abcd
abcd abc


### 3.2. Cắt lát đảo ngược `s[::-1]`

Cú pháp cắt lát là `s[start:stop:step]`. Với `step = -1` ta đi từ cuối về đầu, vì vậy `s[::-1]` là **chuỗi đảo ngược**. Mỗi lần đảo mất `O(độ dài)` thời gian vì phải sao chép.

In [3]:
print("str"[::-1])          # rts
print("abcdef"[::-1])       # fedcba
print(""[::-1] == "")       # True
print("a"[::-1])            # a

rts
fedcba
True
a


### 3.3. `list` — danh sách có thể sửa tại chỗ

Khác với chuỗi, `list` **sửa được**. `lst.append(x)` thêm vào cuối trong `O(1)` (trung bình). `lst.reverse()` đảo ngược **tại chỗ** (không tạo bản sao mới, trả về `None`). `"".join(lst)` ghép list ký tự thành chuỗi.

In [4]:
lst = ["s", "t", "r"]
lst.append("n")
print(lst)                  # ['s', 't', 'r', 'n']
lst.reverse()
print(lst)                  # ['n', 'r', 't', 's']
print("".join(lst))         # nrts
print(lst.reverse())        # None  (reverse không trả về gì)

['s', 't', 'r', 'n']
['n', 'r', 't', 's']
nrts
None


### 3.4. `collections.deque` — hàng đợi hai đầu

`deque` (đọc là "đếch") là cấu trúc cho phép thêm / xoá ở **cả hai đầu** trong `O(1)`: `append(x)` thêm vào **phải**, `appendleft(x)` thêm vào **trái**. Với `list`, chèn vào đầu (`lst.insert(0, x)`) tốn `O(n)` vì phải dời mọi phần tử.

In [5]:
from collections import deque

d = deque()
d.append("b")          # deque(['b'])
d.append("c")          # deque(['b', 'c'])
d.appendleft("a")      # deque(['a', 'b', 'c'])
print(d)
print("".join(d))      # abc
d.appendleft("z")
print("".join(d))      # zabc

deque(['a', 'b', 'c'])
abc
zabc


### 3.5. `str.split(sep)` — tách chuỗi theo dấu phân cách

`"a-b-c".split("-")` trả về `["a", "b", "c"]`. Nếu dấu phân cách đứng liền nhau hoặc ở cuối, ta nhận được phần tử **chuỗi rỗng** `""`. Ta sẽ dùng `s.split("i")` để chia `s` thành các đoạn nằm giữa những chữ `'i'`.

In [6]:
print("string".split("i"))        # ['str', 'ng']
print("poiinter".split("i"))      # ['po', '', 'nter']
print("abi".split("i"))           # ['ab', '']
print("abc".split("i"))           # ['abc']

['str', 'ng']
['po', '', 'nter']
['ab', '']
['abc']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy gõ `s = "poiinter"` bằng tay và ghi lại chuỗi sau mỗi ký tự:

| bước | ký tự | hành động | màn hình |
|---|---|---|---|
| 1 | `p` | nối vào cuối | `p` |
| 2 | `o` | nối vào cuối | `po` |
| 3 | `i` | **đảo ngược** | `op` |
| 4 | `i` | **đảo ngược** | `po` |
| 5 | `n` | nối vào cuối | `pon` |
| 6 | `t` | nối vào cuối | `pont` |
| 7 | `e` | nối vào cuối | `ponte` |
| 8 | `r` | nối vào cuối | `ponter` |

**Nhận xét:** hai lần `'i'` liên tiếp triệt tiêu nhau (đảo hai lần = như cũ). Cách làm trực giác nhất là làm đúng như đề: một chuỗi `screen`, gặp `'i'` thì đảo, còn lại thì nối. Cách này đúng ngay, chỉ chậm nếu chuỗi dài và có nhiều chữ `'i'`.

In [7]:
# Đếm số lần đảo trong "poiinter" và so với việc thực sự phải đảo
s = "poiinter"
print("số chữ 'i':", s.count("i"))     # 2
print("kết quả cuối:", "ponter")

số chữ 'i': 2
kết quả cuối: ponter


## 5. Cách 1 — Brute Force / baseline: mô phỏng bằng chuỗi

**Ý tưởng:** giữ biến `screen` là chuỗi đang hiển thị. Với mỗi ký tự `ch` của `s`:

1. Nếu `ch == 'i'`: `screen = screen[::-1]` (đảo thật).
2. Ngược lại: `screen = screen + ch` (nối vào cuối).

Đây là bản dịch **thẳng** từ đề sang code, rất phù hợp để làm **oracle** (đáp án chuẩn) kiểm các cách sau.

In [8]:
from typing import List

def final_string_naive(s: str) -> str:
    screen = ""
    for ch in s:
        if ch == "i":
            screen = screen[::-1]     # đảo thật, tốn O(độ dài screen)
        else:
            screen = screen + ch      # nối vào cuối
    return screen

In [9]:
print(final_string_naive("string"))      # rtsng
print(final_string_naive("poiinter"))    # ponter
print(final_string_naive("abc"))         # abc  (không có 'i')
print(final_string_naive("aiiib"))       # ?  tự tính: a -> a, i -> a, i -> a, i -> a, b -> ab

rtsng
ponter
abc
ab


**Độ phức tạp:** gọi `n = len(s)`. Mỗi lần gặp `'i'` ta sao chép toàn bộ chuỗi hiện tại (tối đa `n` ký tự), tệ nhất `O(n)` lần ⇒ Time `O(n²)`. Mỗi lần nối `+` cũng có thể sao chép chuỗi. Space `O(n)`.

**Nút thắt:** với `n <= 100` nó nhanh thừa. Nhưng ý tưởng "đảo thật mỗi lần" lãng phí: ta chỉ cần biết chuỗi **đang theo chiều nào**. Nếu `n` tới `10^5`, mô phỏng `O(n²)` sẽ chậm.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — đảo "lười" (lazy reverse).** Thay vì đảo thật chuỗi, ta giữ một **cờ** `reversed_flag` cho biết "chuỗi thật đang được đọc ngược hay xuôi". Gặp `'i'` chỉ cần **đổi cờ** (`O(1)`).

**Đột phá 2 — chỗ thêm ký tự phụ thuộc cờ.** Khi gõ ký tự mới, nó luôn ở **cuối của chuỗi hiển thị**:

- Nếu cờ = xuôi (chưa đảo): hiển thị = lưu trữ, nên thêm vào **phải** của lưu trữ.
- Nếu cờ = ngược: hiển thị = lưu trữ đảo ngược, nên "cuối hiển thị" = **đầu của lưu trữ**, tức thêm vào **trái**.

Dùng `deque` thì cả hai thao tác đều `O(1)`.

**Đột phá 3 — kết quả cuối.** Khi kết thúc, nếu cờ = ngược thì đảo lưu trữ một lần duy nhất rồi `join`. Tổng `O(n)`.

Kiểm tra với `"string"`: `s,t,r` → lưu trữ `[s,t,r]`, cờ xuôi. Gặp `i` → cờ ngược. Gõ `n`: thêm vào **trái** → `[n,s,t,r]`. Gõ `g`: `[g,n,s,t,r]`. Cuối: cờ ngược → đảo → `r,t,s,n,g` = `"rtsng"` ✓.

In [10]:
from collections import deque

store = deque(["s", "t", "r"])
flag = False             # False: xuôi, True: ngược
flag = not flag          # gặp 'i'
for ch in "ng":
    if flag:
        store.appendleft(ch)
    else:
        store.append(ch)
    print(ch, "->", list(store))
print("kết quả:", "".join(reversed(store)) if flag else "".join(store))   # rtsng

n -> ['n', 's', 't', 'r']
g -> ['g', 'n', 's', 't', 'r']
kết quả: rtsng


## 7. Các cách tiếp theo

Ta giảng thêm 3 cách nữa.

### Cách 2 — Dùng `list` và `reverse()` tại chỗ

**Từng đoạn:**

- `res` là list các ký tự (thay vì chuỗi) để `append` nhanh.
- Gặp `'i'`: `res.reverse()` (đảo tại chỗ, không sao chép nhiều lần như chuỗi nhưng vẫn mất `O(len(res))`).
- Cuối cùng `"".join(res)`.

Cách này vẫn là `O(n²)` trong trường hợp xấu nhất nhưng hằng số nhỏ hơn cách 1, và vẫn là mô phỏng thẳng.

In [11]:
def final_string_list(s: str) -> str:
    res = []
    for ch in s:
        if ch == "i":
            res.reverse()
        else:
            res.append(ch)
    return "".join(res)

In [12]:
print(final_string_list("string"))      # rtsng
print(final_string_list("poiinter"))    # ponter

rtsng
ponter


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def finalString(self, s: str) -> str:
        res = []
        for ch in s:
            if ch == "i":
                res.reverse()
            else:
                res.append(ch)
        return "".join(res)
```

**Độ phức tạp:** Time `O(n²)` trường hợp xấu (mỗi `'i'` đảo tối đa `n` phần tử), Space `O(n)`.

### Cách 3 — `deque` + cờ hướng (tối ưu `O(n)`)

**Từng đoạn:**

- `dq = deque()` lưu các ký tự; `rev = False`.
- Gặp `'i'`: `rev = not rev` (`O(1)`).
- Ký tự thường: `rev` thì `dq.appendleft(ch)`, ngược lại `dq.append(ch)`.
- Cuối: nếu `rev` thì đọc `dq` **từ phải sang trái** (`reversed(dq)`), ngược lại đọc bình thường.

In [13]:
from collections import deque

def final_string_deque(s: str) -> str:
    dq = deque()
    rev = False
    for ch in s:
        if ch == "i":
            rev = not rev
        elif rev:
            dq.appendleft(ch)
        else:
            dq.append(ch)
    return "".join(reversed(dq)) if rev else "".join(dq)

In [14]:
print(final_string_deque("string"))      # rtsng
print(final_string_deque("poiinter"))    # ponter
print(final_string_deque("aiiib"))       # ab

rtsng
ponter
ab


**Code nộp LeetCode (Cách 3):**

```python
from collections import deque

class Solution:
    def finalString(self, s: str) -> str:
        dq = deque()
        rev = False
        for ch in s:
            if ch == "i":
                rev = not rev
            elif rev:
                dq.appendleft(ch)
            else:
                dq.append(ch)
        return "".join(reversed(dq)) if rev else "".join(dq)
```

**Độ phức tạp:** mỗi ký tự được xử lý `O(1)`; `join` cuối `O(n)` ⇒ Time `O(n)`, Space `O(n)`.

### Cách 4 — Tách theo `'i'` bằng `split` rồi ghép đoạn

**Từng đoạn:**

- `s.split("i")` cho các đoạn `seg_0, seg_1, ..., seg_k` nằm giữa các chữ `'i'`.
- Trước đoạn đầu chưa có gì. Mỗi lần đi qua một `'i'` thì chuỗi hiện tại đảo ngược, sau đó nối đoạn kế tiếp: `cur = cur[::-1] + seg`.
- Gọn nhưng vẫn là `O(n²)` trong trường hợp xấu; điểm hay là chỉ đảo đúng `k` lần.

In [15]:
def final_string_split(s: str) -> str:
    parts = s.split("i")
    cur = parts[0]
    for seg in parts[1:]:
        cur = cur[::-1] + seg
    return cur

In [16]:
print(final_string_split("string"))      # rtsng
print(final_string_split("poiinter"))    # ponter
print(final_string_split("abiicd"))      # parts = [ab, "", cd] -> ba -> ab -> abcd

rtsng
ponter
abcd


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def finalString(self, s: str) -> str:
        parts = s.split("i")
        cur = parts[0]
        for seg in parts[1:]:
            cur = cur[::-1] + seg
        return cur
```

**Độ phức tạp:** gọi `k` là số chữ `'i'`. Mỗi lần đảo mất `O(độ dài hiện tại)` ⇒ Time `O(k·n)` tệ nhất `O(n²)`; Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 (deque + cờ) trên `s = "poiinter"`. Cột "đọc ra" là chuỗi hiển thị thực sự tại bước đó.

| bước | ký tự | `rev` sau | `dq` (trái → phải) | đọc ra (hiển thị) |
|---|---|---|---|---|
| 1 | `p` | F | `p` | `p` |
| 2 | `o` | F | `p o` | `po` |
| 3 | `i` | **T** | `p o` | `op` |
| 4 | `i` | **F** | `p o` | `po` |
| 5 | `n` | F | `p o n` | `pon` |
| 6 | `t` | F | `p o n t` | `pont` |
| 7 | `e` | F | `p o n t e` | `ponte` |
| 8 | `r` | F | `p o n t e r` | `ponter` |

Bảng thứ hai, trace `"string"`, nơi `rev = T` kéo dài và ký tự đi vào **bên trái**:

| bước | ký tự | `rev` sau | `dq` | đọc ra |
|---|---|---|---|---|
| 1-3 | `s,t,r` | F | `s t r` | `str` |
| 4 | `i` | **T** | `s t r` | `rts` |
| 5 | `n` | T | `n s t r` | `rtsn` |
| 6 | `g` | T | `g n s t r` | `rtsng` |

In [17]:
def trace_deque(s):
    from collections import deque
    dq = deque()
    rev = False
    for step, ch in enumerate(s, 1):
        if ch == "i":
            rev = not rev
            hanh_dong = "đổi cờ rev"
        elif rev:
            dq.appendleft(ch)
            hanh_dong = "appendleft"
        else:
            dq.append(ch)
            hanh_dong = "append"
        hien_thi = "".join(reversed(dq)) if rev else "".join(dq)
        print(f"bước {step}: ch={ch!r}, {hanh_dong:11} | rev={rev!s:5} | dq={list(dq)} | hiển thị={hien_thi!r}")
    return "".join(reversed(dq)) if rev else "".join(dq)

print(trace_deque("poiinter"))

bước 1: ch='p', append      | rev=False | dq=['p'] | hiển thị='p'
bước 2: ch='o', append      | rev=False | dq=['p', 'o'] | hiển thị='po'
bước 3: ch='i', đổi cờ rev  | rev=True  | dq=['p', 'o'] | hiển thị='op'
bước 4: ch='i', đổi cờ rev  | rev=False | dq=['p', 'o'] | hiển thị='po'
bước 5: ch='n', append      | rev=False | dq=['p', 'o', 'n'] | hiển thị='pon'
bước 6: ch='t', append      | rev=False | dq=['p', 'o', 'n', 't'] | hiển thị='pont'
bước 7: ch='e', append      | rev=False | dq=['p', 'o', 'n', 't', 'e'] | hiển thị='ponte'
bước 8: ch='r', append      | rev=False | dq=['p', 'o', 'n', 't', 'e', 'r'] | hiển thị='ponter'
ponter


In [18]:
print(trace_deque("string"))
print("---")
print(trace_deque("aibic"))   # nhiều chữ 'i' xen kẽ

bước 1: ch='s', append      | rev=False | dq=['s'] | hiển thị='s'
bước 2: ch='t', append      | rev=False | dq=['s', 't'] | hiển thị='st'
bước 3: ch='r', append      | rev=False | dq=['s', 't', 'r'] | hiển thị='str'
bước 4: ch='i', đổi cờ rev  | rev=True  | dq=['s', 't', 'r'] | hiển thị='rts'
bước 5: ch='n', appendleft  | rev=True  | dq=['n', 's', 't', 'r'] | hiển thị='rtsn'
bước 6: ch='g', appendleft  | rev=True  | dq=['g', 'n', 's', 't', 'r'] | hiển thị='rtsng'
rtsng
---
bước 1: ch='a', append      | rev=False | dq=['a'] | hiển thị='a'
bước 2: ch='i', đổi cờ rev  | rev=True  | dq=['a'] | hiển thị='a'
bước 3: ch='b', appendleft  | rev=True  | dq=['b', 'a'] | hiển thị='ab'
bước 4: ch='i', đổi cờ rev  | rev=False | dq=['b', 'a'] | hiển thị='ba'
bước 5: ch='c', append      | rev=False | dq=['b', 'a', 'c'] | hiển thị='bac'
bac


## 9. Code nộp LeetCode

Dùng Cách 3 (deque + cờ): `O(n)` và dễ chứng minh.

In [19]:
from typing import List
from collections import deque

class Solution:
    def finalString(self, s: str) -> str:
        dq = deque()
        rev = False
        for ch in s:
            if ch == "i":
                rev = not rev
            elif rev:
                dq.appendleft(ch)
            else:
                dq.append(ch)
        return "".join(reversed(dq)) if rev else "".join(dq)

In [20]:
print(Solution().finalString("string"))      # rtsng
print(Solution().finalString("poiinter"))    # ponter

rtsng
ponter


## 10. Vì sao thuật toán đúng

**Bất biến (invariant).** Sau khi xử lý mỗi tiền tố của `s`, chuỗi hiển thị bằng:

- `dq` đọc từ trái sang phải nếu `rev = False`;
- `dq` đọc từ phải sang trái nếu `rev = True`.

**Chứng minh quy nạp.**

- *Khởi đầu:* `dq` rỗng, hiển thị rỗng ✓.
- *Gặp `'i'`:* đề bảo đảo ngược chuỗi hiển thị `D` thành `reverse(D)`. Ta đổi `rev`, nên cách đọc `dq` đảo chiều, và hiển thị mới = `reverse(D)` ✓. Không đụng tới `dq`.
- *Gặp ký tự thường `c`:* đề bảo nối `c` vào **cuối** hiển thị `D`, thành `D + c`.
  - Nếu `rev = False`, hiển thị = `dq` trái→phải nên thêm `c` vào **phải** của `dq` cho `dq + c` ✓.
  - Nếu `rev = True`, hiển thị = `dq` phải→trái, tức `reverse(dq)`. Cần `reverse(dq) + c = reverse(c + dq)`, nên thêm `c` vào **trái** của `dq` ✓.

Khi kết thúc, ta xuất chuỗi hiển thị theo bất biến, chính là đáp án.

**Thêm:** hai `'i'` liên tiếp đổi cờ hai lần, về như cũ, khớp với việc đảo hai lần là không đổi.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`, `k` = số chữ `'i'`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Chuỗi, đảo thật | `O(n²)` | `O(n)` | mỗi `'i'` và mỗi `+` sao chép |
| 2. List + `reverse()` | `O(n·k)` ≤ `O(n²)` | `O(n)` | `append` rẻ, `reverse` tốn |
| 3. Deque + cờ | `O(n)` | `O(n)` | mỗi ký tự `O(1)` |
| 4. `split` + đảo | `O(n·k)` ≤ `O(n²)` | `O(n)` | chỉ đảo `k` lần |

Với `n <= 100`, mọi cách đều nhanh. Cách 3 là cách duy nhất vẫn chạy tốt khi `n` lớn nhiều lần đảo.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: không có `'i'`; một ký tự; `'i'` ở cuối; nhiều `'i'` liên tiếp (triệt tiêu theo cặp); `'i'` lẻ liên tiếp; toàn ký tự giống nhau; chuỗi toàn `'i'` ngoại trừ chữ đầu (vì `s[0] != 'i'`).

In [21]:
impls = {
    "naive": final_string_naive,
    "list": final_string_list,
    "deque": final_string_deque,
    "split": final_string_split,
    "Solution": Solution().finalString,
}
cases = [
    ("ví dụ 1", "string", "rtsng"),
    ("ví dụ 2", "poiinter", "ponter"),
    ("không có i", "abc", "abc"),
    ("một ký tự", "a", "a"),
    ("i ở cuối", "abci", "cba"),
    ("hai i liên tiếp", "abiic", "abc"),
    ("ba i liên tiếp", "abiiic", "bac"),
    ("i xen kẽ", "aibic", "bac"),
    ("toàn giống nhau", "aaaa", "aaaa"),
    ("a rồi toàn i", "aiiii", "a"),
]
for name, s, exp in cases:
    for k, f in impls.items():
        got = f(s)
        assert got == exp, (name, k, got, exp)
    print("PASS", name, "->", repr(exp))

PASS ví dụ 1 -> 'rtsng'
PASS ví dụ 2 -> 'ponter'
PASS không có i -> 'abc'
PASS một ký tự -> 'a'
PASS i ở cuối -> 'cba'
PASS hai i liên tiếp -> 'abc'
PASS ba i liên tiếp -> 'bac'
PASS i xen kẽ -> 'bac'
PASS toàn giống nhau -> 'aaaa'
PASS a rồi toàn i -> 'a'


Kiểm tay vài ca: `"abci"`: `abc` → đảo `cba` ✓. `"abiiic"`: `ab` → `ba` → `ab` → `ba`, rồi nối `c` thành `bac` ✓. Ca `"aibic"`: `a` → (`i`) `a` → `ab` → (`i`) `ba` → `bac` ✓. Các đáp án mong đợi ở trên đều tính tay, không lấy từ code.

### 12.2. So sánh các cách trên nhiều input ngẫu nhiên có seed

Input hợp lệ theo constraints: độ dài `1..100`, chữ thường, ký tự đầu **khác** `'i'`. Ta chọn bảng chữ nhỏ có nhiều `'i'` (`"abci"`) để các nhánh đảo xuất hiện thường xuyên.

In [22]:
import random
random.seed(2810)
for _ in range(1000):
    n = random.randint(1, 100)
    alphabet = random.choice(["abci", "ai", "abcdefghijklmnopqrstuvwxyz"])
    chars = [random.choice(alphabet) for _ in range(n)]
    if chars[0] == "i":
        chars[0] = "a"      # constraints: s[0] != 'i'
    s = "".join(chars)
    exp = final_string_naive(s)
    for k, f in impls.items():
        assert f(s) == exp, (s, k)
print("1000 chuỗi ngẫu nhiên: mọi cách khớp mô phỏng gốc")

1000 chuỗi ngẫu nhiên: mọi cách khớp mô phỏng gốc


### 12.3. Kiểm tính chất

Các tính chất đúng với mọi `s` hợp lệ: độ dài kết quả = `n − (số chữ 'i')`; kết quả có cùng **đa tập** ký tự (không tính `'i'`) với `s`; nếu số `'i'` chẵn **và** chúng đứng sát nhau thành từng cặp thì kết quả = `s` đã bỏ hết `'i'`.

In [23]:
random.seed(28100)
from collections import Counter
for _ in range(500):
    n = random.randint(1, 100)
    chars = [random.choice("abcdi") for _ in range(n)]
    if chars[0] == "i":
        chars[0] = "d"
    s = "".join(chars)
    r = Solution().finalString(s)
    assert len(r) == n - s.count("i")
    assert Counter(r) == Counter(s.replace("i", ""))
# cặp 'ii' triệt tiêu
for _ in range(200):
    base = "".join(random.choice("abcd") for _ in range(random.randint(1, 20)))
    k = random.randint(0, 5)
    pos = random.randint(1, len(base))        # chèn sau ít nhất 1 ký tự
    s = base[:pos] + "ii" * k + base[pos:]
    assert Solution().finalString(s) == base, (s, base)
print("tính chất đúng")

tính chất đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Ghi cả chữ `'i'` vào chuỗi

In [24]:
def sai_ghi_i(s: str) -> str:
    screen = ""
    for ch in s:
        screen = screen + ch          # sai: luôn nối, kể cả 'i'
        if ch == "i":
            screen = screen[::-1]
    return screen

print("sai :", sai_ghi_i("string"))                 # 'i' bị lẫn vào kết quả
print("đúng:", Solution().finalString("string"))    # rtsng

sai : irtsng
đúng: rtsng


**Vì sao sai:** chữ `'i'` chỉ **kích hoạt** việc đảo, nó không hiện lên màn hình. Phải dùng `if ... elif/else`: nếu `'i'` thì đảo, **ngược lại** mới nối.

### Lỗi 2: Đảo mà không gán lại

In [25]:
def sai_khong_gan(s: str) -> str:
    screen = ""
    for ch in s:
        if ch == "i":
            screen[::-1]              # sai: tạo chuỗi mới nhưng bỏ đi, screen không đổi
        else:
            screen += ch
    return screen

print("sai :", sai_khong_gan("string"))             # strng
print("đúng:", Solution().finalString("string"))    # rtsng

sai : strng
đúng: rtsng


**Vì sao sai:** chuỗi bất biến, `screen[::-1]` **trả về** chuỗi mới chứ không sửa `screen`. Phải gán `screen = screen[::-1]`. Tương tự với list: `lst.reverse()` sửa tại chỗ nhưng `lst = lst.reverse()` gán `None` (lỗi hay gặp).

### Lỗi 3: Đảo toàn bộ `s` một lần cuối thay vì đảo từng tiền tố

In [26]:
def sai_dao_mot_lan(s: str) -> str:
    kq = s.replace("i", "")
    return kq[::-1] if s.count("i") % 2 == 1 else kq   # sai: bỏ qua vị trí chữ 'i'

print("sai :", sai_dao_mot_lan("string"), "| đúng:", Solution().finalString("string"))
print("sai :", sai_dao_mot_lan("poiinter"), "| đúng:", Solution().finalString("poiinter"))
print("sai :", sai_dao_mot_lan("abicd"), "| đúng:", Solution().finalString("abicd"))

sai : gnrts | đúng: rtsng
sai : ponter | đúng: ponter
sai : dcba | đúng: bacd


**Vì sao sai:** `'i'` đảo **chuỗi đã gõ tới thời điểm đó**, không phải chuỗi cuối. Ký tự gõ **sau** `'i'` vẫn được nối vào cuối bình thường rồi lại bị đảo bởi `'i'` sau đó. Chỉ khi tất cả `'i'` đứng ở cuối mới đúng.

### Lỗi 4: Dùng deque nhưng quên đảo ở cuối khi `rev = True`

In [27]:
from collections import deque

def sai_quen_dao_cuoi(s: str) -> str:
    dq = deque()
    rev = False
    for ch in s:
        if ch == "i":
            rev = not rev
        elif rev:
            dq.appendleft(ch)
        else:
            dq.append(ch)
    return "".join(dq)                  # sai: quên reversed khi rev = True

print("sai :", sai_quen_dao_cuoi("string"))          # sai thứ tự
print("đúng:", Solution().finalString("string"))     # rtsng

sai : gnstr
đúng: rtsng


**Vì sao sai:** khi `rev = True`, `dq` đang lưu theo chiều **ngược** so với hiển thị; phải đọc từ phải sang trái (`reversed(dq)`).

### Lỗi 5 (nhỏ): Dùng `list.insert(0, ch)` thay `appendleft`

`lst.insert(0, ch)` đúng kết quả nhưng mỗi lần là `O(n)` (dời cả list), làm tổng `O(n²)`. Dùng `deque.appendleft` để đạt `O(1)` mỗi lần.

In [28]:
lst = ["b", "c"]
lst.insert(0, "a")
print(lst)             # ['a', 'b', 'c']  (đúng nhưng O(n) mỗi lần chèn đầu)

['a', 'b', 'c']


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "đảo ngược toàn bộ rồi tiếp tục thêm vào cuối" ⇒ nghĩ ngay tới **đảo lười**: đổi cờ và thêm vào **đầu / cuối** của `deque`.
- Dấu hiệu dùng `deque`: cần thêm / xoá ở **hai đầu** liên tục.
- Khi ràng buộc nhỏ (`n <= 100`), mô phỏng `s = s[::-1]` luôn đủ; nhưng hãy biết cách `O(n)` để dùng khi `n` lớn.
- Luôn nhớ **gán lại** khi dùng phép trả về chuỗi mới (`s = s[::-1]`), và nhớ `list.reverse()` trả `None`.
- Dùng oracle (mô phỏng thẳng đề) so với cách tối ưu trên input ngẫu nhiên để bắt lỗi đảo sai chiều.

## 15. Biến thể của bài

1. **Ký tự đặc biệt khác:** thay `'i'` bằng một tập ký tự (`'i'`, `'o'` đều đảo).
2. **Đảo chỉ phần đã gõ sau lần đảo trước:** làm đề mới và so sánh.
3. **Hỏi kết quả sau mỗi bước** (trả về list chuỗi hiển thị) thay vì chỉ chuỗi cuối.
4. **`n` rất lớn (`10^5`):** cách 3 chạy được, cách 1 không.
5. **Gõ nhiều ký tự một lúc:** xử lý theo từng đoạn giữa hai chữ `'i'`.
6. **Hỏi ngược:** biết kết quả cuối và dãy thao tác, tìm chuỗi gõ ban đầu (đảo quy trình).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chuỗi, đảo thật | dịch thẳng đề | `O(n²)` | `O(n)` | oracle, người mới |
| 2. List + `reverse()` | `append` rẻ, đảo tại chỗ | `O(n·k)` | `O(n)` | muốn mô phỏng nhưng gọn |
| 3. Deque + cờ | đảo lười, thêm đầu/cuối | `O(n)` | `O(n)` | **code nộp**, `n` lớn |
| 4. `split` + đảo | chia thành đoạn giữa các `'i'` | `O(n·k)` | `O(n)` | code ngắn, ít đảo |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"abc"[::-1]`, `"ab" + "c"`, `"a-i-b".split("i")`, và `list(deque(["a","b"]))` sau khi `appendleft("z")`.

**Bài 2 (tính tay).** Với `s = "abicdie"`, viết chuỗi hiển thị sau mỗi ký tự rồi cho đáp án cuối.

**Bài 3 (điền TODO).** Hoàn thành `final_string_todo(s)` theo Cách 3 (deque + cờ).

In [29]:
def final_string_todo(s):
    from collections import deque
    # TODO 1: dq = deque(), rev = False
    # TODO 2: duyệt từng ch trong s
    # TODO 3: nếu ch == 'i' thì đổi cờ rev
    # TODO 4: ngược lại: rev thì appendleft, không rev thì append
    # TODO 5: return "".join(reversed(dq)) nếu rev, ngược lại "".join(dq)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `final_string_split2(s)` bằng `s.split("i")` mà **không nhìn** code mẫu. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `history(s)` trả về **list** chuỗi hiển thị sau mỗi ký tự (kể cả ký tự `'i'`). (b) Viết `final_string_multi(s, specials)` trong đó mọi ký tự thuộc tập `specials` đều làm đảo chuỗi.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"abc"[::-1] = "cba"`; `"a-i-b".split("i") = ["a-", "-b"]`; `appendleft` thêm vào đầu. Bài 2: gõ `a`, `b`, rồi `i` đảo `ab` thành `ba`...
- **Mức 2 (vừa):** Bài 3: năm dòng đã ghi trong comment TODO. Bài 4: `parts = s.split("i")`, `cur = parts[0]`, rồi mỗi đoạn tiếp theo làm `cur = cur[::-1] + seg`.
- **Mức 3 (sâu):** Bài 5(a): giữ `screen`; tại mỗi bước append bản sao `screen` vào list `history`. Bài 5(b): thay điều kiện `ch == "i"` bằng `ch in specials`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
from collections import deque

# Bài 1
assert "abc"[::-1] == "cba"
assert "a-i-b".split("i") == ["a-", "-b"]
d = deque(["a", "b"]); d.appendleft("z")
assert list(d) == ["z", "a", "b"]

# Bài 2: s = "abicdie"
# a -> a ; b -> ab ; i -> ba ; c -> bac ; d -> bacd ; i -> dcab ; e -> dcabe
assert Solution().finalString("abicdie") == "dcabe"

# Bài 3
def final_string_todo_ans(s):
    dq = deque()
    rev = False
    for ch in s:
        if ch == "i":
            rev = not rev
        elif rev:
            dq.appendleft(ch)
        else:
            dq.append(ch)
    return "".join(reversed(dq)) if rev else "".join(dq)

assert final_string_todo_ans("string") == "rtsng"
assert final_string_todo_ans("poiinter") == "ponter"

# Bài 4
def final_string_split2(s):
    parts = s.split("i")
    cur = parts[0]
    for seg in parts[1:]:
        cur = cur[::-1] + seg
    return cur

assert final_string_split2("string") == "rtsng"
assert final_string_split2("poiinter") == "ponter"

# Bài 5(a)
def history(s):
    screen = ""
    out = []
    for ch in s:
        if ch == "i":
            screen = screen[::-1]
        else:
            screen += ch
        out.append(screen)
    return out

assert history("string") == ["s", "st", "str", "rts", "rtsn", "rtsng"]
assert history("poiinter")[-1] == "ponter"

# Bài 5(b)
def final_string_multi(s, specials):
    screen = ""
    for ch in s:
        if ch in specials:
            screen = screen[::-1]
        else:
            screen += ch
    return screen

assert final_string_multi("string", {"i"}) == "rtsng"
assert final_string_multi("abcode", {"o"}) == "cbade"
# 'a','b','c' -> abc ; 'o' -> cba ; 'd','e' -> cbade

import random
random.seed(1)
for _ in range(300):
    s = "".join(random.choice("abci") for _ in range(random.randint(1, 30)))
    if s[0] == "i":
        s = "a" + s[1:]
    assert final_string_todo_ans(s) == final_string_split2(s) == Solution().finalString(s) == final_string_multi(s, {"i"})
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đảo lười bằng `deque` + cờ hướng)

```python
from typing import List
from collections import deque

class Solution:
    def finalString(self, s: str) -> str:
        dq = deque()
        rev = False
        for ch in s:
            if ch == "i":
                rev = not rev                  # đảo lười: chỉ đổi cờ
            elif rev:
                dq.appendleft(ch)              # cuối hiển thị = đầu lưu trữ
            else:
                dq.append(ch)
        return "".join(reversed(dq)) if rev else "".join(dq)
```

**Checklist**

- [ ] Chữ `'i'` **không** được ghi vào kết quả?
- [ ] Gặp `'i'` thì đảo **tiền tố đã gõ**, không phải toàn bộ `s`?
- [ ] Nhớ gán lại `screen = screen[::-1]` (chuỗi bất biến)?
- [ ] Nếu dùng deque + cờ: nhớ đọc ngược ở cuối khi `rev = True`?
- [ ] Đã thử: không có `'i'`, một ký tự, `'i'` ở cuối, nhiều `'i'` liên tiếp (chẵn / lẻ)?
- [ ] Đã so cách tối ưu với mô phỏng thẳng trên input ngẫu nhiên có seed?
- [ ] Signature đúng `finalString(self, s: str) -> str`?